# ハンズオン②：全店を1つのモデルで予測する — グローバルモデルの需要予測

ハンズオン①の `FORECAST` は、**店ごとの過去の売上の形**から学習しました。手軽で強力ですが、実務では次のような場面で工夫が必要になります。

| 場面 | FORECAST（①） | このハンズオン（グローバルモデル） |
|---|---|---|
| 開業したばかりの店・開業予定の店 | 学習データに十分な履歴がない店は予測できない | **全店を1つのモデル**で学習し、立地・業態・開業からの日数が似た店から学ぶ |
| 店の外の情報 | 日ごとに変わる外生変数（販促・天気など） | 駅の乗降客数・周辺人口など**店の属性**も特徴量にできる（国土交通省の実データ） |
| 検証 | 学習時の評価はオプションで指定するだけ | **期間をずらしたバックテスト**を自分で設計し、**Experiment Tracking** で作り変えの効果を記録・比較する |
| 運用 | SQL で作って SQL で使う | **Model Registry** に登録して SQL/Python から使う |


**データ**：店舗・売上・販促・天気は**合成データ**、立地特徴量（駅・乗降客数・人口）は国土交通省の**実データ**です。事前に `setup/setup.sql` で作っておきます。
**環境**：Container Runtime の Notebook（xgboost / scikit-learn / plotly / snowflake-ml-python はプリインストール済みで、pip は不要です）。

In [ ]:
%%sql -r use_role
USE ROLE SYSADMIN;

In [ ]:
%%sql -r use_wh
USE WAREHOUSE COMPUTE_WH;

In [ ]:
# データ準備の確認：データは事前に setup/setup.sql（ワークシートで Run All）で作ってあります
from snowflake.snowpark.context import get_active_session
session = get_active_session()

try:
    chk = session.table("DF_HANDSON.CONTRACT.CHECK_RESULTS").to_pandas()
    session.table("DF_HANDSON.ML.DAILY").limit(1).collect()
except Exception as e:
    raise RuntimeError("データがまだありません。先に setup/setup.sql をワークシートで実行（Run All）してください") from e

errors = chk[(~chk["PASSED"]) & (chk["SEVERITY"] == "ERROR")]
if len(errors):
    raise RuntimeError("データの契約チェックでエラーがあります\n" + errors.to_string(index=False))
warns = chk[(~chk["PASSED"]) & (chk["SEVERITY"] == "WARN")]
if len(warns):
    print("注意:\n" + warns.to_string(index=False))
print("データは準備できています")

from importlib.metadata import version
print("snowflake-ml-python", version("snowflake-ml-python"), "（Experiment Tracking には 1.19.0 以上が必要）")

In [ ]:
# 地図の準備：地図を描画するときの関数を準備
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

FMT_JA = {"STATION_IN": "駅ナカ", "STATION_FRONT": "駅前", "RESIDENTIAL": "住宅街"}
FMT_COLOR = {"駅ナカ": "#1f77b4", "駅前": "#ff7f0e", "住宅街": "#2ca02c", "不明": "#7f7f7f"}

_wards = session.table("DF_HANDSON.REF.WARD").to_pandas()
WARD_FC = {"type": "FeatureCollection",
           "features": [{"type": "Feature", "id": r.WARD_CODE, "properties": {"ward_code": r.WARD_CODE, "name": r.WARD_NAME},
                         "geometry": json.loads(r.GEOJSON)} for r in _wards.itertuples()]}
RAIL_FC = {"type": "FeatureCollection",
           "features": [{"type": "Feature", "properties": {}, "geometry": json.loads(g)}
                        for g in session.table("DF_HANDSON.REF.RAIL").to_pandas()["GEOJSON"]]}
_sea = {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {}, "geometry":
        {"type": "Polygon", "coordinates": [[[139.4, 35.4], [140.1, 35.4], [140.1, 35.95], [139.4, 35.95], [139.4, 35.4]]]}}]}
BASE_LAYERS = [
    {"source": _sea, "type": "fill", "color": "#e8eef2", "below": "traces"},
    {"source": WARD_FC, "type": "fill", "color": "#f7f5f0", "below": "traces"},
    {"source": WARD_FC, "type": "line", "color": "#9a9a9a", "line": {"width": 0.8}, "below": "traces"},
    {"source": RAIL_FC, "type": "line", "color": "#b0b0b0", "line": {"width": 1.0}, "below": "traces"},
]
MAP_LAYOUT = dict(map_layers=BASE_LAYERS, margin={"l": 0, "r": 0, "t": 40, "b": 0})
CENTER = {"lat": 35.69, "lon": 139.74}

def fmt_ja(s):
    return s.map(FMT_JA).fillna("不明")

def show_map(df, title="", zoom=10.2, **kw):
    """緯度(LAT)・経度(LON)の列を持つ表を、23区の地図に点で表示します"""
    fig = px.scatter_map(df, lat="LAT", lon="LON", map_style="white-bg", center=CENTER, zoom=zoom,
                         height=600, title=title, **kw)
    fig.update_layout(**MAP_LAYOUT)
    fig.show()

## 1. 特徴量：予測する時点で分かる値だけを使う
最大で約1か月先を予測するので、**過去の売上は「35日以上前」のものだけ**を特徴量にします（`ML.GLOBAL_FEATURES` ビュー）。  
「昨日の売上」のような値は、1か月先を予測する時点ではまだ分からないからです（使うと評価が甘くなる＝リーク）。

| グループ（変数名） | 特徴量 |
|---|---|
| カレンダー・販促・天気（`CALENDAR`） | 曜日、祝日、年末年始/お盆、販促、月、季節（sin/cos）、気温、降水量 |
| 過去の売上（`LAGS`） | 5〜8週前の同じ曜日の売上、31〜58日前の平均（新店は NULL のまま） |
| 店の属性（`STORE`） | 業態、開業からの日数 |
| 立地（`LOCATION`、国土交通省の実データ） | 最寄り駅の乗降客数、駅からの距離、1km圏の駅数・乗降客数・人口・高齢化率 |

In [ ]:
import numpy as np
cfg = {k: pd.Timestamp(v) for k, v in session.table("DF_HANDSON.ML.CONFIG").to_pandas().iloc[0].to_dict().items()}
df = session.table("DF_HANDSON.ML.GLOBAL_FEATURES").to_pandas()
df["DS"] = pd.to_datetime(df["DS"])
stores_df = session.table("DF_HANDSON.CONTRACT.STORE").to_pandas()
FMT = stores_df.set_index("STORE_ID")["STORE_FORMAT"].map(FMT_JA).fillna("不明")
NAME = stores_df.set_index("STORE_ID")["STORE_NAME"]

# 特徴量のグループ（次の章で、少しずつ足しながら効果を比べます）
CALENDAR = ["DOW", "IS_HOLIDAY", "IS_SPECIAL", "IS_PROMO", "MONTH", "DOY_SIN", "DOY_COS", "TEMP_AVG", "PRECIP_MM"]
LAGS = ["LAG_35", "LAG_42", "LAG_49", "LAG_56", "SAME_DOW_MEAN_5_8W", "MEAN_28_ENDING_31"]
STORE = ["DAYS_SINCE_OPEN", "FMT_STATION_IN", "FMT_STATION_FRONT", "FMT_RESIDENTIAL"]
LOCATION = ["LOG_STATION_PASSENGERS", "DIST_STATION_M", "N_STATIONS_1KM", "LOG_PASSENGERS_1KM", "LOG_POP_1KM", "POP_65PLUS_RATIO_1KM"]
FEATURES = CALENDAR + LAGS + STORE + LOCATION
assert set(FEATURES) == {c for c in df.columns if c not in ("STORE_ID", "DS", "SALES")}, "GLOBAL_FEATURES の列とグループが一致しません"
df[FEATURES] = df[FEATURES].astype(float)
print(f"行数 {len(df):,}（店 {df['STORE_ID'].nunique()} × 日）、特徴量 {len(FEATURES)} 個")
print(f"テスト月：{cfg['TEST_START']:%Y-%m-%d} 〜 {cfg['TEST_END']:%Y-%m-%d}／未来：{cfg['FUTURE_START']:%Y-%m-%d} 〜 {cfg['FUTURE_END']:%Y-%m-%d}")
df[df["DS"] == cfg["TEST_START"]].head(5)

## 2. バックテスト × Experiment Tracking：特徴量を足すと良くなるか
テスト月を1か月ずつ過去にずらして4回テストし、毎回「その月の前日まで」のデータだけで学習します。
- **グローバルモデル**：XGBoost。全店・全日を1つのモデルで学習（売上は log 変換して学習）
- **ベースライン**：5〜8週前の同じ曜日の平均（予測モデルを使わない場合の目安）
- 指標は **WAPE**（誤差の絶対値の合計 ÷ 実績の合計）。開業120日以内の店だけの WAPE（新店）も見ます

モデルの作り変え（特徴量の追加）を **Snowflake ML Experiments** に run として記録し、効果を比べます。

| run | 使う特徴量 |
|---|---|
| `RUN1_CALENDAR_LAG` | カレンダー・販促・天気 ＋ 過去の売上 |
| `RUN2_ADD_STORE` | ＋ 店の属性（業態・開業からの日数） |
| `RUN3_ADD_LOCATION` | ＋ 立地（国土交通省の実データ：乗降客数・人口） |

記録した run は Snowsight の **AI & ML » Experiments » DEMAND_GLOBAL_XGB** でも比べられます。

In [ ]:
from xgboost import XGBRegressor

PARAMS = dict(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=5, random_state=42, n_jobs=-1)
NEW_STORE_DAYS = 120   # 開業からこの日数以内を「新店」として別に評価する

def wape(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return float(np.abs(y - p).sum() / np.abs(y).sum()) if len(y) else float("nan")

def fit(train, features=None):
    features = features or FEATURES
    return XGBRegressor(**PARAMS).fit(train[features], np.log1p(train["SALES"]))

def predict(model, x, features=None):
    return np.expm1(model.predict(x[features or FEATURES])).clip(min=0)

has_y = df["SALES"].notna()

def backtest(features):
    """テスト月をずらして4回学習・評価します。月ごとの WAPE、最後（k=0）のテスト結果とモデルを返します"""
    rows = []
    for k in (3, 2, 1, 0):                      # k=0 がハンズオン①と同じテスト月
        t_start = cfg["TEST_START"] - pd.DateOffset(months=k)
        t_end = t_start + pd.offsets.MonthEnd(0)
        train = df[has_y & (df["DS"] < t_start)]
        test = df[has_y & (df["DS"] >= t_start) & (df["DS"] <= t_end)].copy()
        model = fit(train, features)
        test["PRED"] = predict(model, test, features)
        test["NAIVE"] = test["SAME_DOW_MEAN_5_8W"].fillna(test["MEAN_28_ENDING_31"])
        ok = test["NAIVE"].notna()             # ベースラインは履歴がある行だけで比べる
        new = test["DAYS_SINCE_OPEN"] <= NEW_STORE_DAYS
        rows.append({"month": f"{t_start:%Y-%m}",
                     "wape": wape(test.loc[ok, "SALES"], test.loc[ok, "PRED"]),
                     "wape_baseline": wape(test.loc[ok, "SALES"], test.loc[ok, "NAIVE"]),
                     "wape_all_rows": wape(test["SALES"], test["PRED"]),
                     "wape_new_store": wape(test.loc[new, "SALES"], test.loc[new, "PRED"])})
    return pd.DataFrame(rows), test, model

Experiment Trackingに記録するときは、モデル学習のコードをラップします。
```python
with exp.start_run():
```
また、記録に残したいメトリクスを明示します。
```python
exp.log_metrics()
```

In [ ]:
from snowflake.ml.experiment import ExperimentTracking

session.use_database("DF_HANDSON"); session.use_schema("ML")
exp = ExperimentTracking(session=session)          # 保存先は DF_HANDSON.ML
exp.set_experiment("DEMAND_GLOBAL_XGB")

RUNS = {"RUN1_CALENDAR_LAG": CALENDAR + LAGS,
        "RUN2_ADD_STORE": CALENDAR + LAGS + STORE,
        "RUN3_ADD_LOCATION": CALENDAR + LAGS + STORE + LOCATION}

results = {}
for run_name, feats in RUNS.items():
    try:
        exp.delete_run(run_name)                 # 再実行時用（同じ名前の run は作り直す）
    except Exception:
        pass
    with [穴埋め](run_name):
        bt, test, model = backtest(feats)
        exp.log_params({**PARAMS, "n_features": len(feats), "features": ",".join(feats)})
        for step, r in enumerate(bt.to_dict("records")):     # step = バックテストの何回目か（古い月から順）
            [穴埋め]({k: v for k, v in r.items() if k != "month" and pd.notna(v)}, step=step)
        [穴埋め]({f"{k}_mean": float(bt[k].mean()) for k in ("wape", "wape_baseline", "wape_all_rows", "wape_new_store")
                         if bt[k].notna().any()}, step=len(bt))
    results[run_name] = {"bt": bt, "test": test, "model": model, "features": feats}
    print(f"✓ {run_name}：特徴量 {len(feats)} 個、平均 WAPE {bt['wape'].mean():.1%}")

In [ ]:
summary = pd.DataFrame([{"run": n, "特徴量の数": len(r["features"]),
                         "WAPE（平均）": r["bt"]["wape"].mean(),
                         "新店の WAPE（平均）": r["bt"]["wape_new_store"].mean(),
                         "新店を含む全行の WAPE（平均）": r["bt"]["wape_all_rows"].mean(),
                         "ベースライン": r["bt"]["wape_baseline"].mean()} for n, r in results.items()])
display(summary.style.format({c: "{:.1%}" for c in summary.columns if "WAPE" in c or c == "ベースライン"}))

by_month = pd.concat([r["bt"].assign(run=n) for n, r in results.items()])
px.line(by_month, x="month", y="wape", color="run", markers=True, title="バックテストの WAPE（run ごと、低いほど良い）",
        labels={"month": "テスト月", "wape": "WAPE"}, height=360).update_yaxes(tickformat=".0%").show()
px.bar(summary.melt(id_vars="run", value_vars=["WAPE（平均）", "新店の WAPE（平均）"], var_name="対象", value_name="WAPE"),
       x="run", y="WAPE", color="対象", barmode="group", title="特徴量を足した効果（全体と新店）",
       height=360).update_yaxes(tickformat=".0%").show()

# 以降は、平均 WAPE が最も低い run を使います
BEST = summary.loc[summary["WAPE（平均）"].idxmin(), "run"]
FEATURES, last, last_model = results[BEST]["features"], results[BEST]["test"], results[BEST]["model"]
rows = results[BEST]["bt"].to_dict("records")
print(f"以降で使う run：{BEST}（特徴量 {len(FEATURES)} 個）")

### FORECAST（ハンズオン①）と同じ12店・同じ月で比べる

In [ ]:
try:
    fc = session.table("DF_HANDSON.ML.FC_RESULT").to_pandas()
    fc["DS"] = pd.to_datetime(fc["DS"])
    cmp = last.merge(fc[["STORE_ID", "DS", "FORECAST"]], on=["STORE_ID", "DS"])
    res = pd.DataFrame({"方法": ["FORECAST（ML関数・店ごと）", "グローバルモデル（XGBoost）", "ベースライン"],
                        "WAPE": [wape(cmp["SALES"], cmp["FORECAST"]), wape(cmp["SALES"], cmp["PRED"]),
                                 wape(cmp["SALES"], cmp["NAIVE"])]})
    print(f"比較した店：{cmp['STORE_ID'].nunique()} 店 × {cmp['DS'].nunique()} 日")
    display(res.style.format({"WAPE": "{:.1%}"}))
except Exception as e:
    print("ハンズオン①の FORECAST の結果（ML.FC_RESULT）がないため、比較をスキップします：", type(e).__name__)

どちらが良いかは**データ次第**です。履歴が十分にある店では FORECAST も強力です。  
グローバルモデルの強みは、次に見る**新店**と、店の属性を使えることにあります。実務では手間とのバランスで使い分けを検討します。

## 3. 新店の予測と、SHAP による説明
前の章で選んだ run のモデルを使います。開業からの日数ごとに誤差を見て、いちばん新しい店の予測をグラフにします。

In [ ]:
t = last.copy()
t["開業からの日数"] = pd.cut(t["DAYS_SINCE_OPEN"], [-1, 90, 365, 10000], labels=["90日以内", "91〜365日", "1年超"])
by_age = (t.groupby("開業からの日数", observed=True)
           .apply(lambda x: pd.Series({"店数": x["STORE_ID"].nunique(), "WAPE": wape(x["SALES"], x["PRED"])}))
           .reset_index())
display(by_age.style.format({"WAPE": "{:.1%}", "店数": "{:.0f}"}))

newest = t.loc[t["DAYS_SINCE_OPEN"].groupby(t["STORE_ID"]).transform("min").idxmin(), "STORE_ID"]
hist = df[(df["STORE_ID"] == newest) & has_y & (df["DS"] <= cfg["TEST_END"])]
x = t[t["STORE_ID"] == newest]
fig = go.Figure([go.Scatter(x=hist["DS"], y=hist["SALES"], name="実績", line={"color": "black"}),
                 go.Scatter(x=x["DS"], y=x["PRED"], name="グローバルモデルの予測", line={"color": "#d62728"})])
fig.update_layout(title=f"いちばん新しい店：{NAME[newest]}（{FMT[newest]}）", height=360, yaxis_title="円")
fig.show()

In [ ]:
import xgboost as xgb
LABEL = {"DOW": "曜日", "IS_HOLIDAY": "祝日", "IS_SPECIAL": "年末年始・お盆", "IS_PROMO": "販促", "MONTH": "月",
         "DOY_SIN": "季節(sin)", "DOY_COS": "季節(cos)", "TEMP_AVG": "気温", "PRECIP_MM": "降水量",
         "DAYS_SINCE_OPEN": "開業からの日数", "FMT_STATION_IN": "業態：駅ナカ", "FMT_STATION_FRONT": "業態：駅前",
         "FMT_RESIDENTIAL": "業態：住宅街", "LOG_STATION_PASSENGERS": "最寄り駅の乗降客数", "DIST_STATION_M": "駅からの距離",
         "N_STATIONS_1KM": "1km圏の駅数", "LOG_PASSENGERS_1KM": "1km圏の乗降客数", "LOG_POP_1KM": "1km圏の人口",
         "POP_65PLUS_RATIO_1KM": "1km圏の高齢化率", "LAG_35": "5週前の売上", "LAG_42": "6週前の売上", "LAG_49": "7週前の売上",
         "LAG_56": "8週前の売上", "SAME_DOW_MEAN_5_8W": "5〜8週前の同曜日平均", "MEAN_28_ENDING_31": "1〜2か月前の平均売上"}
contrib = last_model.get_booster().predict(xgb.DMatrix(last[FEATURES]), pred_contribs=True)[:, :-1]   # 最終列はバイアス
imp = pd.Series(np.abs(contrib).mean(axis=0), index=[LABEL.get(f, f) for f in FEATURES]).sort_values().tail(12)
px.bar(imp, orientation="h", title="平均 |SHAP|（log 売上への寄与）上位12", labels={"value": "", "index": ""},
       height=420).update_layout(showlegend=False).show()

rain = pd.DataFrame({"降水量(mm)": last["PRECIP_MM"].values, "SHAP": contrib[:, FEATURES.index("PRECIP_MM")],
                     "業態": FMT.reindex(last["STORE_ID"]).values})
fig = px.scatter(rain, x="降水量(mm)", y="SHAP", color="業態", color_discrete_map=FMT_COLOR, opacity=0.5,
                 title="雨の効き方は業態で違う（降水量の SHAP。マイナス＝売上を押し下げる）", height=400)
fig.add_hline(y=0, line_dash="dot", line_color="grey")
fig.show()

**ポイント**：SHAP は「この予測で、各特徴量がどれだけ上げ下げしたか」を行ごとに分解します。  
降水量の SHAP を業態で色分けすると、**雨の日の下がり方は住宅街・駅前の路面店ほど大きく、駅ナカは小さい**ことをモデルが学んだと分かります。  
（SHAP は業態と降水量の「組み合わせの効果」を両方の特徴量に分けて配るので、色ごとの差は実際の差より小さめに出ます）  
重要度の棒グラフだけでは分からない「誰に・いつ効くか」を説明できます。

## 4. Model Registry に登録し、SQL から来月を予測する
選んだ run の特徴量で、全期間のデータを使って学習し直したモデルを登録し、**SQL から直接**来月（未来）の売上を予測します。  
10月に**開業予定の店**（売上の履歴なし）も、立地と業態から予測されます。  

モデルの登録にはRegistryのAPIを使います。
```python
reg.log_model()
```

In [ ]:
from snowflake.ml.registry import Registry
final_model = fit(df[has_y])          # 選んだ run の特徴量（FEATURES）で、全期間を使って学習し直す
session.use_database("DF_HANDSON"); session.use_schema("ML")   # log_model は INFORMATION_SCHEMA を参照するため現在の DB が必要
reg = Registry(session=session, database_name="DF_HANDSON", schema_name="ML")
models = reg.show_models()   # モデルが1つも無いときは列の無い DataFrame が返る
if "name" in models.columns and "DEMAND_GLOBAL_XGB" in models["name"].str.upper().tolist():
    reg.delete_model("DEMAND_GLOBAL_XGB")   # 再実行時用
mv = [穴埋め](final_model, model_name="DEMAND_GLOBAL_XGB", version_name="V1",
                   sample_input_data=df.loc[has_y, FEATURES].head(100), target_platforms=["WAREHOUSE"],
                   comment=f"Global XGBoost daily sales model (target=log1p(sales)) for SNOW BAKERY, experiment run {BEST}",
                   metrics={"experiment": "DEMAND_GLOBAL_XGB", "run": BEST,
                            "backtest_wape": {r["month"]: r["wape"] for r in rows}})
sql_feature_cols = ", ".join(f"f.{c}::FLOAT" for c in FEATURES)   # 次の SQL セルで使う列リスト（Jinja で埋め込む）
print(f"登録しました：DEMAND_GLOBAL_XGB / V1（run {BEST}）")

In [ ]:
%%sql -r future_pred
WITH m AS MODEL DF_HANDSON.ML.DEMAND_GLOBAL_XGB VERSION V1
SELECT f.store_id, f.ds,
       ROUND(GREATEST(EXP(m!PREDICT({{ sql_feature_cols }}):output_feature_0::FLOAT) - 1, 0)) AS pred_sales   -- log1p を元に戻す
FROM DF_HANDSON.ML.GLOBAL_FEATURES f
CROSS JOIN DF_HANDSON.ML.CONFIG c
WHERE f.ds BETWEEN c.future_start AND c.future_end
ORDER BY f.store_id, f.ds;

In [ ]:
fp = future_pred.to_pandas()
fp = fp.merge(stores_df[["STORE_ID", "STORE_NAME", "STORE_FORMAT", "LAT", "LON", "AREA_CODE", "AREA_NAME", "OPEN_DATE"]], on="STORE_ID")
ward = fp.groupby(["AREA_CODE", "AREA_NAME"], as_index=False)["PRED_SALES"].sum()
st = fp.groupby(["STORE_ID", "STORE_NAME", "STORE_FORMAT", "LAT", "LON", "OPEN_DATE"], as_index=False)["PRED_SALES"].sum()
st["開業予定"] = pd.to_datetime(st["OPEN_DATE"]) >= cfg["FUTURE_START"]

fig = px.choropleth_map(ward, geojson=WARD_FC, locations="AREA_CODE", featureidkey="properties.ward_code",
                        color="PRED_SALES", hover_name="AREA_NAME", color_continuous_scale="Oranges", opacity=0.6,
                        map_style="white-bg", center=CENTER, zoom=10.2, height=600,
                        title="来月の予測売上（区ごとの合計）と、開業予定の店（★）",
                        labels={"PRED_SALES": "予測売上(円)"})
p = st[st["開業予定"]]
fig.add_trace(go.Scattermap(lat=p["LAT"], lon=p["LON"], mode="markers+text", text=p["STORE_NAME"], textposition="top right",
                            marker={"size": 16, "color": "#d62728", "symbol": "star"}, name="開業予定"))
fig.update_layout(**MAP_LAYOUT)
fig.show()
print(p[["STORE_NAME", "OPEN_DATE", "PRED_SALES"]].rename(columns={"PRED_SALES": "来月の予測売上(開業日以降)"}).to_string(index=False))

### 後片付け（任意）
```sql
DROP DATABASE DF_HANDSON;
USE ROLE ACCOUNTADMIN; DROP INTEGRATION DSDE3_GIT_API;
```